In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from analysis.color_palettes import make_alternate_versions as make_color_palettes
from analysis.experiments_lib import (
    build_hooks, get_ys, get_series_y,
    base_colors, model_name_options, YVAR_LABELS, XVAR_FNS)

In [ ]:
# Config this notebook needs (kept out of the generic lib):
BLOCK_REPR = 'block_representations'    # dataset constant for the block_representations runs
HK = build_hooks()                     # hook-name -> (leaf, metric) table
measured_br = {
    'pythia-1b-deduped':   [0, 1, 3, 5, 8, 10, 13, 15],
    'pythia-6.9b-deduped': [0, 1, 5, 10, 15, 21, 26, 31],
    'OLMo-2-0425-1B':      [0, 1, 3, 5, 8, 10, 13, 15],
    'OLMo-2-1124-7B':      [0, 1, 5, 10, 15, 21, 26, 31],
}
def bnd_(src):
    def bnd_src(prefix, blks, ms):
        return [(src, HK[f'{prefix}{l}_{m.upper()[:2]}'], f'{m} {l} {prefix}')
                for l in blks for m in ms]
    return bnd_src
bnd_br   = bnd_(BLOCK_REPR)
attn_in  = lambda model, ms=['Au']: bnd_br('AI', measured_br[model], ms)
attn_out = lambda model, ms=['Au']: bnd_br('AO', measured_br[model], ms)
mlp_in   = lambda model, ms=['Au']: bnd_br('MI', measured_br[model], ms)
mlp_out  = lambda model, ms=['Au']: bnd_br('MO', measured_br[model], ms)

In [ ]:
# Animated-spectra engine (analysis/spectrum_anim.py): inject this notebook's data
# backend, expose animate_spectra. The notebook drives animations via animate_spectra /
# anim_meancov / anim_blkres (no plot_spectrum needed here).
import importlib
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_series_y=get_series_y, get_ys=get_ys,
             make_color_palettes=make_color_palettes, base_colors=base_colors,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)
animate_spectra = sa.animate_spectra

## Animated spectra — all boundaries, all models
Profile spectra ($p_j=|\langle\hat\mu, v_j\rangle|^2$) and energy-weighted profiles, animated across checkpoints. Two sections mirroring the source notebook:
- **Mean-covariance** — μ inside the *centered* covariance eigenbasis (`acts_mean_metrics`).
- **Block-vs-residual** — the block-output mean vs the input residual it joins (`mean_metrics_blk_vs_res`, at the sub-block node).

One animation per boundary (profile + weighted, labelled). Run a model's cell to view inline; each animation is ~7 MB. Pass `save_dir='analysis/figures/spectrum_anim'` to `anim_meancov`/`anim_blkres` to render mp4s (fire-and-forget) instead of inline. Pass `smooth=<odd window>` (e.g. `anim_meancov('OLMo-2-0425-1B', smooth=11)`) for Savitzky-Golay denoising that keeps peak heights and edges.

In [ ]:
from IPython.display import display

_MC_BNDS = [('MLP out', mlp_out), ('Attn out', attn_out), ('MLP in', mlp_in), ('Attn in', attn_in)]
_BR_BNDS = [('MLP out', mlp_out), ('Attn out', attn_out)]

# natural y-scale per metric (matches the source notebook); override via (name, {'ylog': ...})
_METRIC_YLOG = {'mean_norm': True, 'mean_frac': True, 'rayleigh': True, 'mahalanobis': True,
                'pr': True, 'pr_weighted': True, 'rayleigh_normed': False, 'max_overlap': False,
                'top_overlap': False, 'max_overlap_idx': False, 'centroid_idx': False,
                'centroid_idx_weighted': False}

def _u_meancov(model, bnd):                         # legend label = layer number only
    return [(s, (h[0], 'acts_mean_metrics'), lbl.split()[1]) for s, h, lbl in bnd(model, ['Au'])]

def _u_blkres(model, bnd):
    return [(s, (h[0].rsplit('.', 1)[0], 'mean_metrics_blk_vs_res'), lbl.split()[1])
            for s, h, lbl in bnd(model, ['Au'])]

def _strip(m, u, model):                            # m = name or (name, {opts})
    name, mopts = m if isinstance(m, tuple) else (m, {})
    return (name, u, [model], {'kind': 'strip', 'ylog': _METRIC_YLOG.get(name, True), **mopts})

def _anim_section(model, bnds, u_fn, tag, save_dir=None, metrics=(), **kw):
    for name, bnd in bnds:
        u = u_fn(model, bnd)
        panels = [('profile', u, [model], {'title': f'{name} — profile $p_j$'}),
                  *[_strip(m, u, model) for m in metrics],          # thin band between the two
                  ('profile_weighted', u, [model], {'title': f'{name} — weighted'})]
        opts = dict(xlog=False, ylog=True, model=model, suptitle=f'{tag} — {name} — {model}', **kw)
        safe = f'{tag}_{name}_{model}'.replace(' ', '_')             # save (if any) is a side effect:
        save = f'{save_dir}/{safe}.mp4' if save_dir else None        # always render inline
        display(animate_spectra(panels, save=save, **opts))

def anim_meancov(model, save_dir=None, metrics=(), **kw):
    bnds = [b for b in _MC_BNDS if b[0] != 'MLP in' or 'olmo' in model.lower()]
    _anim_section(model, bnds, _u_meancov, 'Mean-cov', save_dir, metrics=metrics, **kw)

def anim_blkres(model, save_dir=None, metrics=(), **kw):
    _anim_section(model, _BR_BNDS, _u_blkres, 'Blk-vs-res', save_dir, metrics=metrics, **kw)

### pythia-1b-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-1b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-1b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

### pythia-6.9b-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-6.9b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-6.9b-deduped', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

### OLMo-2-0425-1B

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('OLMo-2-0425-1B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('OLMo-2-0425-1B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

### OLMo-2-1124-7B

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('OLMo-2-1124-7B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('OLMo-2-1124-7B', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], smooth=22, peak=3)

## Animated block-mean cosine heatmap (Exp 1.1)

Pairwise cosine between block-output means (block×block, ordered by depth: attn then mlp per layer), animated across checkpoints. Diagonal hidden. `pin_range=True` (default) holds one symmetric colour range across the whole sweep so colours stay comparable; `pin_range=False` rescales each frame for full contrast.

In [ ]:
# Block×block cosine-of-means heatmap, animated across checkpoints (mirrors block_mean_cos).
# pin_range=True -> one symmetric colour range over the whole sweep (stable colorbar);
# pin_range=False -> rescale each frame for full per-frame contrast.
# save_dir (if given) also writes an mp4 in the background — it does NOT change the inline render.
def anim_blockmean_cos(model, save_dir=None, pin_range=True, **kw):
    outs = [(BLOCK_REPR, HK[f'{p}{l}_AU'], f'{sub} {l}')          # attn then mlp, by depth
            for l in measured_br[model] for p, sub in [('AO', 'attn'), ('MO', 'mlp')]]
    panel = ('acts_mean_vec', outs, [model],
             {'kind': 'heatmap', 'title': 'Block-mean cosine', 'per_frame': not pin_range})
    opts = dict(ncols=1, model=model, suptitle=f'Block-mean cosine — {model}', **kw)
    save = f'{save_dir}/blockmean_cos_{model}.mp4' if save_dir else None
    display(animate_spectra([panel], save=save, **opts))

In [ ]:
anim_blockmean_cos('pythia-1b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_blockmean_cos('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_blockmean_cos('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

In [ ]:
anim_blockmean_cos('OLMo-2-1124-7B', save_dir='analysis/figures/animations')

## Animated block↔block coupling (samples run)

Pairwise block-output coupling matrices from the `block_representations_samples` runs (every layer), animated across checkpoints: **CKA** (shared subspace, unsigned), **signed trace** (net reinforce/cancel, energy-weighted), **mean per-token cosine** (democratic counterpart). Covariance-level sequel to the block-mean cosine animation above; matrices are stored per checkpoint, so `kind='matrix'` reads them directly.

In [ ]:
# Stored (M,M) coupling matrices, animated. cka is unsigned in [0,1] (static range);
# signed trace / mean cos get the symmetric dynamic range (pin_range as above).
BLOCK_SAMPLES = 'block_representations_samples'

def anim_block_coupling(model, save_dir=None, pin_range=True, **kw):
    hook = ('', 'block_block_coupling')
    leaves = get_ys(BLOCK_SAMPLES, model, hook, 'leaves')[0][0]
    labels = [l.removeprefix('blk').removesuffix('.out') for l in leaves]
    panels = [(y, [(BLOCK_SAMPLES, hook)], [model],
               {'kind': 'matrix', 'labels': labels, 'title': t, 'per_frame': not pin_range, **o})
              for y, t, o in [('cka', 'CKA', {'dynamic': False, 'vmin': 0, 'vmax': 1}),
                              ('signed_trace', 'signed trace', {}),
                              ('mean_cos', 'mean cos', {})]]
    opts = dict(ncols=3, model=model, suptitle=f'Block↔block coupling — {model}', **kw)
    save = f'{save_dir}/block_coupling_{model}.mp4' if save_dir else None
    display(animate_spectra(panels, save=save, **opts))

In [ ]:
anim_block_coupling('pythia-1b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_block_coupling('pythia-6.9b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_block_coupling('OLMo-2-0425-1B', save_dir='analysis/figures/animations')

In [ ]:
anim_block_coupling('OLMo-2-1124-7B', save_dir='analysis/figures/animations')